# Notebook 02 – Data Cleaning
## Echo Chambers in Climate Change Discussions on Reddit

This notebook takes the raw JSONL files from `data/raw/` and:
1. Loads and merges all files
2. Removes bots, deleted users, and suspected spam accounts
3. Parses timestamps and adds time features
4. Resolves parent comment → parent author (critical for building edges!)
5. Saves a clean Parquet file to `data/processed/comments_clean.parquet`

**Output:** `data/processed/comments_clean.parquet` — each row is one directed edge (author → parent_author)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.cleaner import DataCleaner

print("✅ Imports successful")

## Configuration

In [ ]:
SUBREDDITS   = ["climate", "climatechange", "environment", "Collapse"]
RAW_DIR      = "../data/raw"
PROCESSED_DIR = "../data/processed"

cleaner = DataCleaner(raw_dir=RAW_DIR, processed_dir=PROCESSED_DIR)
print("DataCleaner ready")

## Step 1: Load Raw Comments

In [ ]:
raw_comments = cleaner.load_comments(SUBREDDITS)
print(f"\nRaw comments loaded: {len(raw_comments):,} rows")
print(f"Columns: {list(raw_comments.columns)}")
raw_comments.head(3)

## Step 2: Run Cleaning Pipeline

In [ ]:
clean_comments = cleaner.clean(raw_comments)
print(f"\nCleaned: {len(clean_comments):,} rows")
print(f"Date range: {clean_comments['datetime'].min()} → {clean_comments['datetime'].max()}")
clean_comments.dtypes

## Step 3: Load Posts & Build ID → Author Map

In [ ]:
raw_posts = cleaner.load_posts(SUBREDDITS)

posts_for_map = pd.DataFrame()
if not raw_posts.empty and 'id' in raw_posts.columns:
    posts_for_map = raw_posts[['id', 'author']].dropna()

id_to_author = cleaner.build_id_to_author(clean_comments, posts_for_map)
print(f"ID→Author map size: {len(id_to_author):,}")

## Step 4: Resolve Parent Authors (Build Edge Table)

In [ ]:
df = cleaner.add_parent_author(clean_comments, id_to_author)

print(f"Edge table rows  : {len(df):,}")
print(f"Unique authors   : {df['author'].nunique():,}")
print(f"Unique parent auth: {df['parent_author'].nunique():,}")
print(f"Subreddits       : {df['subreddit'].value_counts().to_dict()}")
df[['author', 'parent_author', 'created_utc', 'year_month', 'subreddit']].head()

## Step 5: Exploratory Data Analysis

In [ ]:
# ── 5a. Comments per month ──
monthly_counts = df.groupby('year_month').size().reset_index(name='count')
monthly_counts['period_str'] = monthly_counts['year_month'].astype(str)

plt.figure(figsize=(16, 4), facecolor='#0f1117')
ax = plt.gca()
ax.set_facecolor('#1a1d27')
ax.bar(range(len(monthly_counts)), monthly_counts['count'], color='#00e5ff', alpha=0.8)
ax.set_xticks(range(0, len(monthly_counts), 3))
ax.set_xticklabels(monthly_counts['period_str'].iloc[::3], rotation=45, ha='right', color='#aaa')
ax.set_title('Reply Comments per Month', color='white', fontsize=14)
ax.set_ylabel('Count', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(axis='y', color='#2d2d3a', linestyle='--', alpha=0.5)
for spine in ax.spines.values(): spine.set_color('#444')
plt.tight_layout()
plt.savefig('../figures/eda_monthly_counts.png', dpi=120, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 5b. Top 20 most active users ──
top_users = df['author'].value_counts().head(20)

plt.figure(figsize=(12, 5), facecolor='#0f1117')
ax = plt.gca()
ax.set_facecolor('#1a1d27')
ax.bar(top_users.index, top_users.values, color='#51cf66', alpha=0.8)
plt.xticks(rotation=45, ha='right', color='#aaa')
ax.set_title('Top 20 Most Active Users (after bot removal)', color='white', fontsize=13)
ax.set_ylabel('Number of comments', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(axis='y', color='#2d2d3a', linestyle='--', alpha=0.5)
for spine in ax.spines.values(): spine.set_color('#444')
plt.tight_layout()
plt.savefig('../figures/eda_top_users.png', dpi=120, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 5c. Subreddit distribution ──
sub_counts = df['subreddit'].value_counts()
plt.figure(figsize=(8, 5), facecolor='#0f1117')
ax = plt.gca()
ax.set_facecolor('#1a1d27')
colors = ['#00e5ff', '#ff6b6b', '#51cf66', '#ffd43b', '#a855f7']
ax.pie(sub_counts.values, labels=sub_counts.index, colors=colors[:len(sub_counts)],
       autopct='%1.1f%%', textprops={'color': 'white'}, startangle=90)
ax.set_title('Edge Distribution by Subreddit', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/eda_subreddit_pie.png', dpi=120, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## Step 6: Save Processed Data

In [ ]:
# Save to Parquet (efficient, preserves dtypes)
out_path = Path(PROCESSED_DIR) / 'comments_clean.parquet'
df.to_parquet(out_path, index=False)
print(f"✅ Saved → {out_path}")
print(f"   Shape  : {df.shape}")
print(f"   Size   : {out_path.stat().st_size / 1e6:.1f} MB")

# Preview
df[['author', 'parent_author', 'year_month', 'subreddit', 'score']].describe(include='all')

## ✅ Done!
Proceed to **`03_network_building.ipynb`**.